In [1]:
%%bash
set -euo pipefail
chmod +x scripts/download_311_2025_01.sh
./scripts/download_311_2025_01.sh 2>&1 | tee logs/download_311_2025_01.log


Total rows in 2025-01: 348180
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2462k    0 2462k    0     0   652k      0 --:--:--  0:00:03 --:--:--  652k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2459k    0 2459k    0     0   478k      0 --:--:--  0:00:05 --:--:--  637k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2460k    0 2460k    0     0   744k      0 --:--:--  0:00:03 --:--:--  744k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2444k    0 2444k    0     0   518k      0 --:--:--  0:00:04 --:--:--  518k
  % Total    % Receive

In [2]:
%%bash
set -euo pipefail

cat > scripts/download_taxi_2025_01.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail

OUTDIR="data_raw/taxi"
mkdir -p "$OUTDIR"

URL="https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
OUT="${OUTDIR}/yellow_tripdata_2025-01.parquet"

echo "Downloading taxi parquet -> ${OUT}"
curl -L --retry 5 --retry-delay 2 "$URL" -o "$OUT"

ls -lh "$OUT"
EOF

chmod +x scripts/download_taxi_2025_01.sh


In [3]:
%%bash
set -euo pipefail
./scripts/download_taxi_2025_01.sh 2>&1 | tee logs/download_taxi_2025_01.log


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 56.4M  100 56.4M    0     0  8056k      0  0:00:07  0:00:07 --:--:-- 8088k
-rw-r--r--  1 jingchang  staff    56M Feb 27 11:38 data_raw/taxi/yellow_tripdata_2025-01.parquet


In [4]:
%%bash
set -euo pipefail

cat > scripts/download_zone_lookup.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail

OUTDIR="data_raw/lookup"
mkdir -p "$OUTDIR"

URL="https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
OUT="${OUTDIR}/taxi_zone_lookup.csv"

echo "Downloading zone lookup -> ${OUT}"
curl -L --retry 5 --retry-delay 2 "$URL" -o "$OUT"

ls -lh "$OUT"
head -n 3 "$OUT"
EOF

chmod +x scripts/download_zone_lookup.sh
./scripts/download_zone_lookup.sh 2>&1 | tee logs/download_zone_lookup.log


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12331  100 12331    0     0   194k      0 --:--:-- --:--:-- --:--:--  194k
-rw-r--r--  1 jingchang  staff    12K Feb 27 11:39 data_raw/lookup/taxi_zone_lookup.csv
"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"


In [5]:
%%bash
set -euo pipefail

cat > pipeline.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail

mkdir -p logs

echo "[1/4] Download 311 (2025-01)"
./scripts/download_311_2025_01.sh 2>&1 | tee logs/step1_311.log

echo "[2/4] Download Taxi (2025-01)"
./scripts/download_taxi_2025_01.sh 2>&1 | tee logs/step2_taxi.log

echo "[3/4] Download Taxi Zone Lookup"
./scripts/download_zone_lookup.sh 2>&1 | tee logs/step3_lookup.log

echo "[4/4] Spark ETL (TODO)"
echo "TODO: spark-submit spark_jobs/etl_*.py ..."

echo "DONE: Ingest pipeline finished."
EOF

chmod +x pipeline.sh


In [6]:
%%bash
set -euo pipefail
./pipeline.sh


[1/4] Download 311 (2025-01)
Total rows in 2025-01: 348180
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2462k    0 2462k    0     0   411k      0 --:--:--  0:00:05 --:--:--  607k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2459k    0 2459k    0     0   561k      0 --:--:--  0:00:04 --:--:--  561k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2460k    0 2460k    0     0   409k      0 --:--:--  0:00:06 --:--:--  568k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2444k    0 2444k    0     0   369k      0 --:--:--  0:00:06 --:--:--

In [7]:
%%bash
set -euo pipefail
echo "311:"
ls -lh data_raw/311 | head
echo "Taxi:"
ls -lh data_raw/taxi
echo "Lookup:"
ls -lh data_raw/lookup


311:
total 108672
-rw-r--r--@ 1 jingchang  staff   2.4M Feb 27 11:40 311_2025_01_part1.csv
-rw-r--r--  1 jingchang  staff   2.4M Feb 27 11:41 311_2025_01_part10.csv
-rw-r--r--  1 jingchang  staff   2.4M Feb 27 11:41 311_2025_01_part11.csv
-rw-r--r--  1 jingchang  staff   2.4M Feb 27 11:41 311_2025_01_part12.csv
-rw-r--r--  1 jingchang  staff   2.4M Feb 27 11:41 311_2025_01_part13.csv
-rw-r--r--  1 jingchang  staff   2.4M Feb 27 11:41 311_2025_01_part14.csv
-rw-r--r--  1 jingchang  staff   2.4M Feb 27 11:41 311_2025_01_part15.csv
-rw-r--r--  1 jingchang  staff   2.4M Feb 27 11:41 311_2025_01_part16.csv
-rw-r--r--  1 jingchang  staff   2.4M Feb 27 11:41 311_2025_01_part17.csv
Taxi:
total 116864
-rw-r--r--  1 jingchang  staff    56M Feb 27 11:42 yellow_tripdata_2025-01.parquet
Lookup:
total 32
-rw-r--r--  1 jingchang  staff    12K Feb 27 11:42 taxi_zone_lookup.csv


In [8]:
%%bash
set -euo pipefail

cat > scripts/download_311_2023_2025.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail

DATASET="erm2-nwe9"
OUTDIR="data_raw/311"
mkdir -p "$OUTDIR"

SELECT="unique_key,created_date,complaint_type,descriptor,borough,latitude,longitude"

LIMIT=20000

for YEAR in 2023 2024 2025; do
  for MONTH in $(seq -w 1 12); do
    START="${YEAR}-${MONTH}-01T00:00:00"
    if [[ "$MONTH" == "12" ]]; then
      NEXT_YEAR=$((YEAR+1)); NEXT_MONTH="01"
    else
      NEXT_YEAR=$YEAR; NEXT_MONTH=$(printf "%02d" $((10#$MONTH+1)))
    fi
    END="${NEXT_YEAR}-${NEXT_MONTH}-01T00:00:00"

    WHERE="created_date between '${START}' and '${END}'"
    WHERE_ENC=$(echo "$WHERE" | sed 's/ /%20/g')

    # count
    N=$(curl -sL "https://data.cityofnewyork.us/resource/${DATASET}.json?\$select=count(1)%20as%20n&\$where=${WHERE_ENC}" \
      | python3 -c "import sys,json; print(json.load(sys.stdin)[0]['n'])")

    echo "==> ${YEAR}-${MONTH}: ${N} rows"
    if [[ "$N" == "0" ]]; then
      continue
    fi

    OFFSET=0
    PART=1
    while [[ $OFFSET -lt $N ]]; do
      OUT="${OUTDIR}/311_${YEAR}_${MONTH}_part${PART}.csv"
      if [[ -f "$OUT" ]]; then
        echo "    exists, skip: $OUT"
        OFFSET=$((OFFSET + LIMIT))
        PART=$((PART + 1))
        continue
      fi

      URL="https://data.cityofnewyork.us/resource/${DATASET}.csv?\$select=${SELECT}&\$where=${WHERE_ENC}&\$order=created_date&\$limit=${LIMIT}&\$offset=${OFFSET}"
      echo "    downloading part ${PART} offset=${OFFSET}"
      curl -L --retry 5 --retry-delay 2 "$URL" -o "$OUT"

      OFFSET=$((OFFSET + LIMIT))
      PART=$((PART + 1))
      sleep 0.3
    done
  done
done

echo "Done downloading 311 2023-2025."
EOF

chmod +x scripts/download_311_2023_2025.sh
echo "created scripts/download_311_2023_2025.sh"


created scripts/download_311_2023_2025.sh


In [9]:
%%bash
set -euo pipefail
./scripts/download_311_2023_2025.sh 2>&1 | tee logs/download_311_2023_2025.log


==> 2023-01: 242176 rows
    downloading part 1 offset=0
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2456k    0 2456k    0     0  1077k      0 --:--:--  0:00:02 --:--:-- 1077k
    downloading part 2 offset=20000
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2455k    0 2455k    0     0   421k      0 --:--:--  0:00:05 --:--:--  587k
    downloading part 3 offset=40000
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2455k    0 2455k    0     0   713k      0 --:--:--  0:00:03 --:--:--  713k
    downloading part 4 offset=60000
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Uplo

CalledProcessError: Command 'b'set -euo pipefail\n./scripts/download_311_2023_2025.sh 2>&1 | tee logs/download_311_2023_2025.log\n'' returned non-zero exit status 56.

In [10]:
%%bash
cp scripts/download_311_2023_2025.sh scripts/download_311_2023_2025.sh.bak
echo "backup created"


backup created


In [11]:
%%bash
set -euo pipefail

cat > scripts/download_311_2023_2025.sh <<'EOF'
#!/usr/bin/env bash
set -u  
set -o pipefail

DATASET="erm2-nwe9"
OUTDIR="data_raw/311"
mkdir -p "$OUTDIR"

SELECT="unique_key,created_date,complaint_type,descriptor,borough,latitude,longitude"

LIMIT=20000
SLEEP_SEC=1.0

curl_get () {
  local url="$1"
  local out="$2"
  curl -L \
    --retry 12 --retry-all-errors --retry-delay 3 \
    --connect-timeout 20 --max-time 300 \
    -H "Accept: text/csv" \
    "$url" -o "$out"
}

for YEAR in 2023 2024 2025; do
  for MONTH in $(seq -w 1 12); do
    START="${YEAR}-${MONTH}-01T00:00:00"
    if [[ "$MONTH" == "12" ]]; then
      NEXT_YEAR=$((YEAR+1)); NEXT_MONTH="01"
    else
      NEXT_YEAR=$YEAR; NEXT_MONTH=$(printf "%02d" $((10#$MONTH+1)))
    fi
    END="${NEXT_YEAR}-${NEXT_MONTH}-01T00:00:00"

    WHERE="created_date between '${START}' and '${END}'"
    WHERE_ENC=$(echo "$WHERE" | sed 's/ /%20/g')

    COUNT_URL="https://data.cityofnewyork.us/resource/${DATASET}.json?\$select=count(1)%20as%20n&\$where=${WHERE_ENC}"
    N=$(curl -sL --connect-timeout 20 --max-time 60 \
      --retry 8 --retry-all-errors --retry-delay 2 \
      "$COUNT_URL" | python3 -c "import sys,json; print(json.load(sys.stdin)[0]['n'])" 2>/dev/null || echo "ERR")

    if [[ "$N" == "ERR" ]]; then
      echo "==> ${YEAR}-${MONTH}: count failed, skip month"
      continue
    fi

    echo "==> ${YEAR}-${MONTH}: ${N} rows"
    if [[ "$N" == "0" ]]; then
      continue
    fi

    OFFSET=0
    PART=1
    while [[ $OFFSET -lt $N ]]; do
      OUT="${OUTDIR}/311_${YEAR}_${MONTH}_part${PART}.csv"


      if [[ -s "$OUT" ]]; then
        echo "    exists, skip: $OUT"
        OFFSET=$((OFFSET + LIMIT))
        PART=$((PART + 1))
        continue
      fi

      URL="https://data.cityofnewyork.us/resource/${DATASET}.csv?\$select=${SELECT}&\$where=${WHERE_ENC}&\$order=created_date&\$limit=${LIMIT}&\$offset=${OFFSET}"
      echo "    downloading part ${PART} offset=${OFFSET}"

      TMP="${OUT}.tmp"
      rm -f "$TMP"

      if curl_get "$URL" "$TMP"; then
        if [[ ! -s "$TMP" ]]; then
          echo "    WARNING: empty response, will retry later"
          rm -f "$TMP"
        else
          mv "$TMP" "$OUT"
        fi
      else
        echo "    ERROR: curl failed (likely rate limit / reset). Will retry later."
        rm -f "$TMP"
        sleep 5
      fi

      OFFSET=$((OFFSET + LIMIT))
      PART=$((PART + 1))
      sleep "$SLEEP_SEC"
    done
  done
done

echo "Done downloading 311 2023-2025."
EOF

chmod +x scripts/download_311_2023_2025.sh
echo "patched scripts/download_311_2023_2025.sh"


patched scripts/download_311_2023_2025.sh


In [12]:
%%bash
set -euo pipefail
cp scripts/download_311_2023_2025.sh scripts/download_311_2023_01_test.sh
sed -i.bak 's/for YEAR in 2023 2024 2025;/for YEAR in 2023;/' scripts/download_311_2023_01_test.sh
sed -i.bak 's/for MONTH in $(seq -w 1 12);/for MONTH in 01;/' scripts/download_311_2023_01_test.sh
chmod +x scripts/download_311_2023_01_test.sh

./scripts/download_311_2023_01_test.sh 2>&1 | tee logs/download_311_2023_01_test.log


==> 2023-01: 242176 rows
    exists, skip: data_raw/311/311_2023_01_part1.csv
    exists, skip: data_raw/311/311_2023_01_part2.csv
    exists, skip: data_raw/311/311_2023_01_part3.csv
    exists, skip: data_raw/311/311_2023_01_part4.csv
    exists, skip: data_raw/311/311_2023_01_part5.csv
    exists, skip: data_raw/311/311_2023_01_part6.csv
    exists, skip: data_raw/311/311_2023_01_part7.csv
    exists, skip: data_raw/311/311_2023_01_part8.csv
    exists, skip: data_raw/311/311_2023_01_part9.csv
    exists, skip: data_raw/311/311_2023_01_part10.csv
    exists, skip: data_raw/311/311_2023_01_part11.csv
    exists, skip: data_raw/311/311_2023_01_part12.csv
    exists, skip: data_raw/311/311_2023_01_part13.csv
Done downloading 311 2023-2025.


In [13]:
%%bash
./scripts/download_311_2023_2025.sh 2>&1 | tee -a logs/download_311_2023_2025.log
# 311

==> 2023-01: 242176 rows
    exists, skip: data_raw/311/311_2023_01_part1.csv
    exists, skip: data_raw/311/311_2023_01_part2.csv
    exists, skip: data_raw/311/311_2023_01_part3.csv
    exists, skip: data_raw/311/311_2023_01_part4.csv
    exists, skip: data_raw/311/311_2023_01_part5.csv
    exists, skip: data_raw/311/311_2023_01_part6.csv
    exists, skip: data_raw/311/311_2023_01_part7.csv
    exists, skip: data_raw/311/311_2023_01_part8.csv
    exists, skip: data_raw/311/311_2023_01_part9.csv
    exists, skip: data_raw/311/311_2023_01_part10.csv
    exists, skip: data_raw/311/311_2023_01_part11.csv
    exists, skip: data_raw/311/311_2023_01_part12.csv
    exists, skip: data_raw/311/311_2023_01_part13.csv
==> 2023-02: 225003 rows
    exists, skip: data_raw/311/311_2023_02_part1.csv
    exists, skip: data_raw/311/311_2023_02_part2.csv
    exists, skip: data_raw/311/311_2023_02_part3.csv
    exists, skip: data_raw/311/311_2023_02_part4.csv
    exists, skip: data_raw/311/311_2023_02_pa

In [14]:
%%bash
set -euo pipefail

cat > scripts/download_taxi_2023_2025.sh <<'EOF'
#!/usr/bin/env bash
set -u
set -o pipefail

OUTDIR="data_raw/taxi"
mkdir -p "$OUTDIR"

curl_get () {
  local url="$1"
  local out="$2"
  curl -L --retry 10 --retry-all-errors --retry-delay 2 \
    --connect-timeout 20 --max-time 600 \
    "$url" -o "$out"
}

for YEAR in 2023 2024 2025; do
  for MONTH in $(seq -w 1 12); do
    FILE="yellow_tripdata_${YEAR}-${MONTH}.parquet"
    OUT="${OUTDIR}/${FILE}"
    URL="https://d37ci6vzurychx.cloudfront.net/trip-data/${FILE}"

    if [[ -s "$OUT" ]]; then
      echo "exists, skip: $OUT"
      continue
    fi

    echo "downloading: $FILE"
    TMP="${OUT}.tmp"
    rm -f "$TMP"
    if curl_get "$URL" "$TMP"; then
      mv "$TMP" "$OUT"
      ls -lh "$OUT"
    else
      echo "FAILED: $URL"
      rm -f "$TMP"
    fi
    sleep 0.2
  done
done

echo "Done downloading taxi 2023-2025."
EOF

chmod +x scripts/download_taxi_2023_2025.sh
echo "created scripts/download_taxi_2023_2025.sh"


created scripts/download_taxi_2023_2025.sh


In [15]:
%%bash
set -euo pipefail
mkdir -p data_raw/taxi
curl -L --retry 10 --retry-all-errors --retry-delay 2 \
  --connect-timeout 20 --max-time 600 \
  "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet" \
  -o "data_raw/taxi/yellow_tripdata_2023-01.parquet"
ls -lh data_raw/taxi/yellow_tripdata_2023-01.parquet


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 45.4M  100 45.4M    0     0  9327k      0  0:00:04  0:00:04 --:--:--  9.9M


-rw-r--r--  1 jingchang  staff    45M Feb 27 15:29 data_raw/taxi/yellow_tripdata_2023-01.parquet


In [16]:
%%bash
./scripts/download_taxi_2023_2025.sh 2>&1 | tee logs/download_taxi_2023_2025.log


exists, skip: data_raw/taxi/yellow_tripdata_2023-01.parquet
downloading: yellow_tripdata_2023-02.parquet
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 45.5M  100 45.5M    0     0  8270k      0  0:00:05  0:00:05 --:--:-- 9633k
-rw-r--r--  1 jingchang  staff    46M Feb 27 15:30 data_raw/taxi/yellow_tripdata_2023-02.parquet
downloading: yellow_tripdata_2023-03.parquet
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 53.5M  100 53.5M    0     0  8249k      0  0:00:06  0:00:06 --:--:-- 8282k
-rw-r--r--  1 jingchang  staff    54M Feb 27 15:30 data_raw/taxi/yellow_tripdata_2023-03.parquet
downloading: yellow_tripdata_2023-04.parquet
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   

In [17]:
!pip -q install meteostat


DEPRECATION: pyodbc 4.0.0-unsupported has a non-standard version number. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pyodbc or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scipy 1.6.2 requires numpy<1.23.0,>=1.16.5, but you have numpy 1.24.4 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
from datetime import datetime
from meteostat import Point, Hourly
import pandas as pd
import os

os.makedirs("data_raw/weather", exist_ok=True)

nyc = Point(40.7829, -73.9654)

start = datetime(2023, 1, 1)
end   = datetime(2025, 12, 31, 23, 59)

dfw = Hourly(nyc, start, end).fetch().reset_index()
dfw.to_parquet("data_raw/weather/weather_hourly_nyc_2023_2025.parquet", index=False)

dfw.head(), dfw.shape


(                 time  temp  dwpt  rhum  prcp  snow   wdir  wspd  wpgt  \
 0 2023-01-01 00:00:00  10.0   8.9  93.0   1.1  <NA>  185.0   6.5  <NA>   
 1 2023-01-01 01:00:00  11.0   9.7  92.0   1.4  <NA>  186.0   7.9  <NA>   
 2 2023-01-01 02:00:00  12.0  11.5  97.0   0.5  <NA>  223.0  10.4  <NA>   
 3 2023-01-01 03:00:00  12.0  11.5  97.0   0.7  <NA>  230.0   9.0  <NA>   
 4 2023-01-01 04:00:00  12.0  11.5  97.0   1.3  <NA>  237.0  10.4  <NA>   
 
      pres  tsun  coco  
 0  1011.0  <NA>   9.0  
 1  1010.0  <NA>   9.0  
 2  1010.0  <NA>   8.0  
 3  1009.0  <NA>   8.0  
 4  1009.0  <NA>   9.0  ,
 (26304, 12))

In [20]:
%%bash
ls -lh data_raw/weather
du -sh data_raw/weather


total 824
-rw-r--r--  1 jingchang  staff   411K Feb 27 16:00 weather_hourly_nyc_2023_2025.parquet
412K	data_raw/weather


In [21]:
%%bash
du -sh data_raw/311 data_raw/taxi data_raw/weather data_raw/lookup


1.4G	data_raw/311
2.0G	data_raw/taxi
412K	data_raw/weather
 16K	data_raw/lookup
